In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (
    Input, Dense, Dropout, BatchNormalization, Activation,
    Embedding, Conv1D, SpatialDropout1D,
    Bidirectional, LSTM,
    GlobalMaxPooling1D, GlobalAveragePooling1D,
    Concatenate, Multiply, Softmax, Lambda
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.regularizers import l2

from sklearn.metrics import (
    precision_recall_curve, auc, roc_curve,
    confusion_matrix
)

In [2]:
seq_rdic = ['A','I','L','V','F','W','Y','N','C','Q','M','S','T','D','E','R','H','K','G','P','O','U','X','B','Z']
seq_dic = {w: i+1 for i,w in enumerate(seq_rdic)}

def encodeSeq(seq):
    if pd.isnull(seq):
        return [0]
    return [seq_dic.get(aa, 0) for aa in seq]

In [3]:
def parse_data(dti_path, drug_path, protein_path,
               prot_len=2500, drug_len=2048, with_label=True):

    protein_df = pd.read_csv(protein_path, index_col="Protein_ID")
    drug_df = pd.read_csv(drug_path, index_col="Compound_ID")
    dti_df = pd.read_csv(dti_path)

    protein_df["encoded"] = protein_df.Sequence.map(encodeSeq)

    df = pd.merge(dti_df, protein_df, left_on="Protein_ID", right_index=True)
    df = pd.merge(df, drug_df, left_on="Compound_ID", right_index=True)

    protein_feature = pad_sequences(df["encoded"], maxlen=prot_len)
    drug_feature = np.stack(df["morgan_fp_r2"].map(lambda x: x.split("\t"))).astype(float)

    result = {
        "protein_feature": protein_feature,
        "drug_feature": drug_feature,
        "Compound_ID": df["Compound_ID"].values,
        "Protein_ID": df["Protein_ID"].values
    }

    if with_label:
        result["label"] = df["Label"].values

    return result

In [4]:
##################################################################################################

# def build_lstm_model(prot_len=2500, drug_len=2048,
#                      embed_dim=64, rnn_units=128):

#     # ------------------
#     # Inputs
#     # ------------------
#     input_p = Input(shape=(prot_len,))
#     input_d = Input(shape=(drug_len,))

#     # ------------------
#     # Protein branch (LSTM)
#     # ------------------
#     x = Embedding(input_dim=26, output_dim=embed_dim)(input_p)

#     x = Bidirectional(LSTM(rnn_units, return_sequences=True))(x)

#     # Pooling (important)
#     x_max = GlobalMaxPooling1D()(x)
#     x_avg = GlobalAveragePooling1D()(x)

#     x = Concatenate()([x_max, x_avg])

#     x = Dense(128, activation='relu')(x)
#     x = Dropout(0.3)(x)

#     # ------------------
#     # Drug branch
#     # ------------------
#     d = Dense(512, activation='relu')(input_d)
#     d = Dropout(0.2)(d)
#     d = Dense(128, activation='relu')(d)

#     # ------------------
#     # Combine
#     # ------------------
#     combined = Concatenate()([x, d])

#     z = Dense(256, activation='relu')(combined)
#     z = Dropout(0.3)(z)
#     z = Dense(64, activation='relu')(z)

#     output = Dense(1, activation='sigmoid')(z)

#     model = Model(inputs=[input_p, input_d], outputs=output)

#     model.compile(
#         optimizer=Adam(1e-4),
#         loss='binary_crossentropy',
#         metrics=['accuracy']
#     )

#     return model


def build_lstm_model(prot_len=2500, drug_len=2048, embed_dim=128):

    from tensorflow.keras.layers import (
        Input, Dense, Dropout, Embedding,
        LSTM, Bidirectional,
        GlobalMaxPooling1D, GlobalAveragePooling1D,
        Concatenate, LayerNormalization
    )
    from tensorflow.keras.models import Model
    from tensorflow.keras.optimizers import Adam

    # Inputs
    input_p = Input(shape=(prot_len,))
    input_d = Input(shape=(drug_len,))

    # Protein branch (Improved LSTM)
    x = Embedding(input_dim=26, output_dim=embed_dim)(input_p)

    x = Bidirectional(LSTM(
        128,
        return_sequences=True,
        dropout=0.3,
        recurrent_dropout=0.2
    ))(x)

    x = LayerNormalization()(x)

    x = Bidirectional(LSTM(
        128,
        return_sequences=True,
        dropout=0.3
    ))(x)

    # Pooling
    x_max = GlobalMaxPooling1D()(x)
    x_avg = GlobalAveragePooling1D()(x)
    x = Concatenate()([x_max, x_avg])

    x = Dense(128, activation='relu')(x)
    x = Dropout(0.3)(x)

    # Drug branch
    d = Dense(512, activation='relu')(input_d)
    d = Dropout(0.3)(d)
    d = Dense(128, activation='relu')(d)

    # Combine
    combined = Concatenate()([x, d])

    z = Dense(256, activation='relu')(combined)
    z = Dropout(0.3)(z)
    z = Dense(64, activation='relu')(z)

    output = Dense(1, activation='sigmoid')(z)

    model = Model(inputs=[input_p, input_d], outputs=output)

    model.compile(
        optimizer=Adam(1e-4),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [5]:
train_data = parse_data(
    "/content/training_dti.csv",
    "/content/training_compound.csv",
    "/content/training_protein.csv"
)

val_data = parse_data(
    "/content/validation_dti.csv",
    "/content/validation_compound.csv",
    "/content/validation_protein.csv"
)

In [6]:
model = build_lstm_model()
model.summary()

model.fit(
    [train_data["protein_feature"], train_data["drug_feature"]],
    train_data["label"],
    validation_data=(
        [val_data["protein_feature"], val_data["drug_feature"]],
        val_data["label"]
    ),
    epochs=10,
    batch_size=32
)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 2500)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 2500, 128) │      3,328 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional       │ (None, 2500, 256) │    263,168 │ embedding[0][0]   │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 2500, 256) │        512 │ bidirectional[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 2500, 256) │    394,240 │ layer_normalizat… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 256)       │          0 │ bidirectional_1[… │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ bidirectional_1[… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 2048)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 512)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 512)       │  1,049,088 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     65,664 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 128)       │     65,664 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 256)       │          0 │ dropout[0][0],    │
│ (Concatenate)       │                   │            │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 256)       │     65,792 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 64)        │     16,448 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 1)         │         65 │ dense_4[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 1,923,969 (7.34 MB)

 Trainable params: 1,923,969 (7.34 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 54s 54s/step - accuracy: 1.0000 - loss: 0.6148 - val_accuracy: 0.5000 - val_loss: 0.7013
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 28s 28s/step - accuracy: 0.9524 - loss: 0.5594 - val_accuracy: 0.5000 - val_loss: 0.7110
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - accuracy: 0.9524 - loss: 0.5517 - val_accuracy: 0.5000 - val_loss: 0.7220
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 27s 27s/step - accuracy: 0.9524 - loss: 0.4954 - val_accuracy: 0.5000 - val_loss: 0.7365
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 28s 28s/step - accuracy: 1.0000 - loss: 0.4333 - val_accuracy: 0.5000 - val_loss: 0.7541
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 29s 29s/step - accuracy: 1.0000 - loss: 0.4167 - val_accuracy: 0.5000 - val_loss: 0.7750
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 26s 26s/step - accuracy: 1.0000 - loss: 0.3794 - val_accuracy: 0.5000 - val_loss: 0.7997
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 29s 29s/step - accuracy: 1.0000 - loss: 0.3476 - val_accuracy: 0.5000 - val_loss: 0.8277


In [7]:
model.save("dti_model.h5")

In [8]:
test_data = parse_data(
    "/content/test_dti.csv",
    "/content/test_compound.csv",
    "/content/test_protein.csv"
)

#model = load_model("dti_model.h5", safe_mode=False)

preds = model.predict([
    test_data["protein_feature"],
    test_data["drug_feature"]
]).flatten()

1/1 ━━━━━━━━━━━━━━━━━━━━ 6s 6s/step


In [9]:
result_df = pd.DataFrame({
    ("test", "predicted"): preds,
    ("test", "Compound_ID"): test_data["Compound_ID"],
    ("test", "Protein_ID"): test_data["Protein_ID"],
    ("test", "label"): test_data["label"]
})

result_df.columns = pd.MultiIndex.from_tuples(result_df.columns)
result_df.to_csv("predictions.csv", index=False)

print("Saved predictions.csv")

Saved predictions.csv


In [10]:
def evaluate(preds, labels, threshold=0.5):

    pred_labels = (preds >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(labels, pred_labels).ravel()

    sen = tp / (tp + fn)
    spe = tn / (tn + fp)
    pre = tp / (tp + fp)
    acc = (tp + tn) / (tp + tn + fp + fn)
    f1 = 2 * sen * pre / (sen + pre)

    fpr, tpr, _ = roc_curve(labels, preds)
    auc_score = auc(fpr, tpr)

    precision, recall, _ = precision_recall_curve(labels, preds)
    aupr = auc(recall, precision)

    print("Sensitivity:", sen)
    print("Specificity:", spe)
    print("Accuracy:", acc)
    print("Precision:", pre)
    print("F1:", f1)
    print("AUC:", auc_score)
    print("AUPR:", aupr)

evaluate(preds, test_data["label"])

Sensitivity: 1.0
Specificity: 0.0
Accuracy: 0.5
Precision: 0.5
F1: 0.6666666666666666
AUC: 0.9800000000000001
AUPR: 0.9825757575757575
